In [1]:
import json
import os

os.environ["SPS_HOME"] = "/Users/z5114326/Documents/GitHub/python-fsps/src/fsps/libfsps"

import dynesty
import fsps
import gc_utils
import h5py
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fsps.filters import FILTERS
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm
from sklearn.mixture import GaussianMixture


In [2]:
# sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]
sim_lst = ["m12i"]
sim_dir = "/Users/z5114326/Documents/simulations/"

band_type = "JC"  # "SDSS" or "JC"
# bands = ["sdss_g", "sdss_z"]
bands = ["v"]

snap = 600
min_mass = 1e4

In [3]:
compute_vega_mags = band_type == "JC"

sp = fsps.StellarPopulation(
    imf_type=2,
    zcontinuous=1,
    compute_vega_mags=compute_vega_mags,
)

sp.params["add_dust_emission"] = False
sp.params["add_neb_continuum"] = False
sp.params["add_neb_emission"] = False
sp.params["add_agb_dust_model"] = False
sp.params["dust1"] = 0.0
sp.params["dust2"] = 0.0
sp.params["fcstar"] = 0.0
sp.params["agb_dust"] = 0.0
sp.params["tpagb_norm_type"] = 0

In [4]:
public_snapshot_file = sim_dir + "snapshot_times_public.txt"
pub_data = pd.read_table(public_snapshot_file, comment="#", header=None, sep=r"\s+")
pub_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
timez0 = float(pub_data[pub_data["index"] == snap]["time_Gyr"].values[0])
pub_snaps = np.array(pub_data["index"])

all_data_fil = sim_dir + "/" + "m12i" + "/" + "m12i" + "_res7100/" + "snapshot_times.txt"
all_data = pd.read_table(all_data_fil, comment="#", header=None, sep=r"\s+")
all_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
all_times = np.array(all_data["time_Gyr"])
all_snaps = np.array(all_data["index"])

snap_times = {
    gc_utils.snapshot_name(s): float(pub_data.loc[pub_data["index"] == s, "time_Gyr"].values[0])
    for s in pub_snaps
}

In [5]:
sim_dict = {}

# ------------------------------------------------
# precompute snapshot times once
# ------------------------------------------------
snap_times = {
    gc_utils.snapshot_name(s): float(pub_data.loc[pub_data["index"] == s, "time_Gyr"].values[0])
    for s in pub_snaps
}

timez0 = snap_times[gc_utils.snapshot_name(snap)]

for sim in sim_lst:
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    sim_dict[sim] = {}

    with h5py.File(ghost_file, "r") as ghost_data:
        for it_id, it_grp in ghost_data.items():
            src = it_grp["source"]

            # ---------------------------
            # read once
            # ---------------------------
            grpid = src["grpid"][()]
            amsk = grpid == 0

            logm_tfor = src["logm_tfor"][()]
            logm_tz0 = src["logm_tz0"][()]

            m_tfo = 10.0**logm_tfor
            m_tfo_ev = 0.55 * m_tfo  # stellar-evolution corrected mass

            m_tz0 = np.zeros_like(logm_tz0, dtype=float)
            msk = logm_tz0 != -1
            m_tz0[msk] = 10.0 ** logm_tz0[msk]

            tfor = src["tfor"][()]
            tdis = src["tdis"][()]
            tacc = src["tacc"][()]
            feh = src["feh"][()]
            gcids = src["gcid"][()]

            tdis = np.where(tdis == -1, np.inf, tdis)

            rz0 = np.linalg.norm(src["pxyz_snap600"][()], axis=1)

            # ---------------------------
            # vectorized ages
            # ---------------------------
            age = timez0 - tfor
            age_p = np.where(np.isfinite(tdis), tdis - tfor, age)

            t_tz0 = src["torb_600"][()]
            s_tz0 = src["s_flag"][()].astype(bool)
            sa_flag = src["sa_flag"][()].astype(bool)

            # ---------------------------
            # snapshot mass table
            #   + tfor (m_tfo_ev)
            #   + tdis (0)
            # ---------------------------
            n_gc = len(gcids)
            n_snap = len(pub_snaps)

            mass_arr = np.full((n_gc, n_snap + 2), np.nan)
            time_arr = np.full((n_gc, n_snap + 2), np.nan)

            # ---- formation point ----
            mass_arr[:, 0] = m_tfo_ev
            time_arr[:, 0] = tfor

            # ---- simulation snapshots ----
            for j, (snap_id, time) in enumerate(snap_times.items(), start=1):
                snp = it_grp["snapshots"][snap_id]
                snp_gcids = snp["gcid"][()]
                snp_mass = 10.0 ** snp["logm"][()]

                common, idx_snp, idx_src = np.intersect1d(snp_gcids, gcids, return_indices=True)

                mass_arr[idx_src, j] = snp_mass[idx_snp]
                time_arr[idx_src, j] = time

                # ---- disruption point (only if disrupted) ----
                has_disrupted = np.isfinite(tdis)

                mass_arr[has_disrupted, -1] = 0.0
                time_arr[has_disrupted, -1] = tdis[has_disrupted]

            # ---------------------------
            # final mass cut
            # ---------------------------
            m_tfo_msk = m_tfo >= min_mass

            sim_dict[sim][it_id] = {
                "gcid": gcids[m_tfo_msk],
                "feh": feh[m_tfo_msk],
                "tfor": tfor[m_tfo_msk],
                "tdis": tdis[m_tfo_msk],
                "tacc": tacc[m_tfo_msk],
                "age": age[m_tfo_msk],
                "age_p": age_p[m_tfo_msk],
                "rz0": rz0[m_tfo_msk],
                "t_tz0": t_tz0[m_tfo_msk],
                "s_tz0": s_tz0[m_tfo_msk],
                "sa_flag": sa_flag[m_tfo_msk],
                "acc": amsk[m_tfo_msk],
                # ---- masses ----
                "m_tfo": m_tfo[m_tfo_msk],
                "m_tfo_ev": m_tfo_ev[m_tfo_msk],
                "m_tz0": m_tz0[m_tfo_msk],
                # ---- time-resolved mass history ----
                "m_snp": mass_arr[m_tfo_msk],
                "t_snp": time_arr[m_tfo_msk],
            }

In [6]:
sim_dict[sim][it_id]["acc"]

array([ True,  True,  True, ..., False, False, False], shape=(12339,))

In [7]:
sim_dict[sim][it_id]["tacc"]

array([-1.   , -1.   , -1.   , ..., 13.531, 13.643, 13.643],
      shape=(12339,))

In [8]:
def build_mass_interpolators(dat):
    interps = []

    for m, t in zip(dat["m_snp"], dat["t_snp"]):
        msk = np.isfinite(m) & (m > 0)
        if np.sum(msk) < 2:
            interps.append(None)
            continue

        order = np.argsort(t[msk])
        ts = t[msk][order]
        ms = m[msk][order]

        interps.append(interp1d(ts, ms, kind="linear", bounds_error=False, fill_value=(ms[0], 0.0)))

    return interps


for sim in sim_lst:
    for it_id in sim_dict[sim]:
        sim_dict[sim][it_id]["mass_interp"] = build_mass_interpolators(sim_dict[sim][it_id])

In [9]:
# ============================================================
# BUILD TIME-DEPENDENT GC MAGNITUDES INTO ssp_dict
# ============================================================

dt = 0.2
tims = np.arange(0.4, timez0, dt)
tims = np.append(tims, timez0)

print("\n================ BUILDING ssp_dict ================\n")

bands_tag = "_".join(bands)
n_band = len(bands)

ssp_dict = {}

for sim in sim_lst:
    print(f"[SIM] {sim}")

    # --------------------------------------------------------
    # Collect (age, feh) pairs needed for THIS SIM
    # --------------------------------------------------------
    ages_all = []
    fehs_all = []

    for it_id, dat in sim_dict[sim].items():
        tfor = dat["tfor"]
        feh = dat["feh"]

        for t in tims:
            age_t = t - tfor
            msk = age_t > 0
            if np.any(msk):
                ages_all.append(age_t[msk])
                fehs_all.append(feh[msk])

    ages_all = np.concatenate(ages_all)
    fehs_all = np.concatenate(fehs_all)

    ssp_params = np.column_stack((ages_all, fehs_all))
    unique_ssps = np.unique(ssp_params, axis=0)

    print(f"Unique SSPs needed: {len(unique_ssps)}")

    # --------------------------------------------------------
    # FSPS cache (per sim, per band system)
    # --------------------------------------------------------
    cache_dir = f"../data/ssp_caches/{sim}"
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = f"{cache_dir}/ssp_{band_type}_{bands_tag}.npz"

    if os.path.exists(cache_path):
        print("  → Loading FSPS cache")
        cache = np.load(cache_path)
        ssp_mag = cache["ssp_mag"]
        ssp_mass = cache["ssp_mass"]
        cached_ssps = cache["unique_ssps"]

        if not np.array_equal(cached_ssps, unique_ssps):
            print("SSP grid mismatch — recomputing")
            recompute = True
        else:
            recompute = False
    else:
        recompute = True

    if recompute:
        print("Computing FSPS SSP grid")
        n_ssp = len(unique_ssps)

        ssp_mag = np.empty((n_ssp, n_band))
        ssp_mass = np.empty(n_ssp)

        for i, (age, feh) in enumerate(unique_ssps):
            if i % 100 == 0 or i == n_ssp - 1:
                print(f"    SSP {i + 1}/{n_ssp}")

            sp.params["logzsol"] = feh
            ssp_mag[i] = sp.get_mags(tage=age, bands=bands)
            ssp_mass[i] = sp.stellar_mass

        np.savez(
            cache_path,
            unique_ssps=unique_ssps,
            ssp_mag=ssp_mag,
            ssp_mass=ssp_mass,
        )

    # --------------------------------------------------------
    # Build fast lookup: (age, feh) -> SSP index
    # --------------------------------------------------------
    ssp_lookup = {(unique_ssps[i, 0], unique_ssps[i, 1]): i for i in range(len(unique_ssps))}

    # --------------------------------------------------------
    # Allocate output dictionary for this sim
    # --------------------------------------------------------
    ssp_dict[sim] = {"times": np.asarray(tims)}

    # --------------------------------------------------------
    # Compute magnitudes for each it_id
    # --------------------------------------------------------
    for it_id, dat in sim_dict[sim].items():
        print(f"    [ITER] {it_id}")

        gcids = dat["gcid"]
        tfor = dat["tfor"]
        feh = dat["feh"]
        tdis = dat["tdis"]
        tacc = dat["tacc"]
        mass_interp = dat["mass_interp"]
        insitu = dat["acc"]
        tacc = dat["tacc"]

        n_gc = len(gcids)
        n_t = len(tims)

        mags = np.full((n_gc, n_t, n_band), np.nan, dtype=float)

        for ti, t in enumerate(tims):
            alive = (tfor <= t) & (t < tdis) & (tacc <= t)

            for i in np.where(alive)[0]:
                fmass = mass_interp[i]
                if fmass is None:
                    continue

                m_now = float(fmass(t))
                if m_now <= 0:
                    continue

                age = t - tfor[i]
                if age <= 0:
                    continue

                ssp_idx = ssp_lookup.get((age, feh[i]))
                if ssp_idx is None:
                    continue

                mags[i, ti, :] = ssp_mag[ssp_idx] - 2.5 * np.log10(m_now / ssp_mass[ssp_idx])

        ssp_dict[sim][it_id] = {
            "gcid": gcids,
            "mags": mags,  # (N_gc, N_time, N_band)
            "insitu": insitu,
            "tacc": tacc,
        }

print("\n================ ssp_dict COMPLETE ================\n")


================ BUILDING ssp_dict ================

[SIM] m12i
Unique SSPs needed: 5595775
Computing FSPS SSP grid
    SSP 1/5595775
    SSP 101/5595775
    SSP 201/5595775
    SSP 301/5595775
    SSP 401/5595775
    SSP 501/5595775
    SSP 601/5595775
    SSP 701/5595775
    SSP 801/5595775
    SSP 901/5595775
    SSP 1001/5595775
    SSP 1101/5595775
    SSP 1201/5595775
    SSP 1301/5595775
    SSP 1401/5595775
    SSP 1501/5595775
    SSP 1601/5595775
    SSP 1701/5595775
    SSP 1801/5595775
    SSP 1901/5595775
    SSP 2001/5595775
    SSP 2101/5595775
    SSP 2201/5595775
    SSP 2301/5595775
    SSP 2401/5595775
    SSP 2501/5595775
    SSP 2601/5595775
    SSP 2701/5595775
    SSP 2801/5595775
    SSP 2901/5595775
    SSP 3001/5595775
    SSP 3101/5595775
    SSP 3201/5595775
    SSP 3301/5595775
    SSP 3401/5595775
    SSP 3501/5595775
    SSP 3601/5595775
    SSP 3701/5595775
    SSP 3801/5595775
    SSP 3901/5595775
    SSP 4001/5595775
    SSP 4101/5595775
    SSP 4201/

In [10]:
def gif_wrapping(
    plot_func,
    param_name,
    param_values,
    gif_path,
    fixed_kwargs=None,
    interval=200,
    dpi=150,
    figsize=(8, 6),
):
    """
    Generic animation wrapper.
    """

    fixed_kwargs = fixed_kwargs or {}

    fig, ax = plt.subplots(figsize=figsize)

    def update(value):
        kwargs = fixed_kwargs.copy()
        kwargs[param_name] = value
        plot_func(ax=ax, **kwargs)
        return (ax,)

    anim = FuncAnimation(fig, update, frames=param_values, interval=interval, blit=False)

    anim.save(gif_path, writer=PillowWriter(fps=1000 // interval), dpi=dpi)
    plt.close(fig)

In [23]:
def mag_dict(ax, sim, ssp_dict, tim):
    ax.clear()
    bins = np.arange(-15, 0)

    hist_dict = {"sum": None, "n": 0}

    tims = ssp_dict[sim]["times"]
    tidx = np.where(tims == tim)[0][0]

    for it_id in ssp_dict[sim].keys():
        # for it_id in ["it099"]:
        if it_id == "times":
            continue

        cols = []
        for mags_j in ssp_dict[sim][it_id]["mags"]:
            col = mags_j[tidx][0]
            cols.append(col)
        cols = np.asarray(cols)
        hssp, _ = np.histogram(cols, bins=bins)

        if hist_dict["sum"] is None:
            hist_dict["sum"] = hssp.astype(float)
        else:
            hist_dict["sum"] += hssp

        hist_dict["n"] += 1

    # bin_centers = 0.5 * (bins[:-1] + bins[1:])
    # widths = bins[1:] - bins[:-1]

    binsum = hist_dict["sum"] / hist_dict["n"]

    ax.step(bins[:-1], binsum, where="post", color="k")
    # ax.fill_between(bins[:-1], binsum, step="post", alpha=0.5, color="r")

    ax.set_xlabel(r"M$_{V}$")
    ax.set_ylabel("Number")

    ax.set_xlim(0, -15)
    ax.set_ylim(0)

    text = sim + "\n" + "t = " + str(np.round(tim, 2)) + " Gyr"

    ax.text(
        0.04,
        0.94,
        text,
        transform=ax.transAxes,
        color="k",
        ha="left",
        va="center",
        bbox=dict(
            facecolor="white",
            edgecolor="white",
        ),
    )

In [ ]:
def cols_hist(ax, sim, ssp_dict, tim):
    ax.clear()
    bins = np.arange(-2, 2.1, 0.1)

    hist_dict = {"sum": None, "n": 0}

    tims = ssp_dict[sim]["times"]
    tidx = np.where(tims == tim)[0][0]

    for it_id in ssp_dict[sim].keys():
        # for it_id in ["it099"]:
        if it_id == "times":
            continue

        cols = []
        for mags_j in ssp_dict[sim][it_id]["mags"]:
            col = mags_j[tidx][0] - mags_j[tidx][1]
            cols.append(col)
        cols = np.asarray(cols)
        hssp, _ = np.histogram(cols, bins=bins)

        if hist_dict["sum"] is None:
            hist_dict["sum"] = hssp.astype(float)
        else:
            hist_dict["sum"] += hssp

        hist_dict["n"] += 1

    # bin_centers = 0.5 * (bins[:-1] + bins[1:])
    # widths = bins[1:] - bins[:-1]

    binsum = hist_dict["sum"] / hist_dict["n"]

    ax.step(bins[:-1], binsum, where="post", color="k")
    # ax.fill_between(bins[:-1], binsum, step="post", alpha=0.5, color="r")

    ax.set_xlabel(r"(g - z)$_{0}$")
    ax.set_ylabel("Number")

    ax.set_xlim(-2, 2)
    ax.set_ylim(0)

    text = sim + "\n" + "t = " + str(np.round(tim, 2)) + " Gyr"

    ax.text(
        0.04,
        0.94,
        text,
        transform=ax.transAxes,
        color="k",
        ha="left",
        va="center",
        bbox=dict(
            facecolor="white",
            edgecolor="white",
        ),
    )


def cols_hist_origin(ax, sim, ssp_dict, tim):
    ax.clear()
    bins = np.arange(-2, 2.1, 0.1)

    hist_dict = {"insum": None, "exsum": None, "n": 0}

    tims = ssp_dict[sim]["times"]
    tidx = np.where(tims == tim)[0][0]

    for it_id in ssp_dict[sim].keys():
        # for it_id in ["it099"]:
        if it_id == "times":
            continue

        insitu = ssp_dict[sim][it_id]["insitu"]
        taccs = ssp_dict[sim][it_id]["tacc"]

        cols_in = []
        cols_ex = []
        for j, mags_j in enumerate(ssp_dict[sim][it_id]["mags"]):
            col = mags_j[tidx][0] - mags_j[tidx][1]

            inflag = insitu[j]
            tacc = taccs[j]
            if inflag:
                cols_in.append(col)
            else:
                # don't include GCs that haven't yet been accreted
                if tim < tacc:
                    continue
                else:
                    cols_ex.append(col)

        cols_in = np.asarray(cols_in)
        cols_ex = np.asarray(cols_ex)

        hssp_in, _ = np.histogram(cols_in, bins=bins)
        hssp_ex, _ = np.histogram(cols_ex, bins=bins)

        if hist_dict["insum"] is None:
            hist_dict["insum"] = hssp_in.astype(float)
            hist_dict["exsum"] = hssp_ex.astype(float)
        else:
            hist_dict["insum"] += hssp_in
            hist_dict["exsum"] += hssp_ex

        hist_dict["n"] += 1

    # bin_centers = 0.5 * (bins[:-1] + bins[1:])
    # widths = bins[1:] - bins[:-1]

    binsum_in = hist_dict["insum"] / hist_dict["n"]
    binsum_ex = hist_dict["exsum"] / hist_dict["n"]

    ax.step(bins[:-1], binsum_in, where="post", color="r", label="in-situ")
    ax.step(bins[:-1], binsum_ex, where="post", color="b", label="ex-situ")
    # ax.fill_between(bins[:-1], binsum, step="post", alpha=0.5, color="r")

    ax.set_xlabel(r"(g - z)$_{0}$")
    ax.set_ylabel("Number")

    ax.set_xlim(-2, 2)
    ax.set_ylim(0)

    ax.legend(loc="upper right")

    text = sim + "\n" + "t = " + str(np.round(tim, 2)) + " Gyr"

    ax.text(
        0.04,
        0.94,
        text,
        transform=ax.transAxes,
        color="k",
        ha="left",
        va="center",
        bbox=dict(
            facecolor="white",
            edgecolor="white",
        ),
    )

In [ ]:
run_gif = False

sim = "m12m"
if run_gif:
    gif_path = "../data/results/" + sim + "_tcolor.gif"
    gif_wrapping(
        plot_func=cols_hist,
        param_name="tim",
        param_values=tims,
        gif_path=gif_path,
        fixed_kwargs=dict(sim=sim, ssp_dict=ssp_dict),
        interval=250,
    )

In [28]:
run_gif = True

sim = "m12i"
if run_gif:
    gif_path = "../data/results/" + sim + "_tMv.gif"
    gif_wrapping(
        plot_func=mag_dict,
        param_name="tim",
        param_values=tims,
        gif_path=gif_path,
        fixed_kwargs=dict(sim=sim, ssp_dict=ssp_dict),
        interval=250,
    )

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

tim = tims[-1]
cols_hist(ax, sim, ssp_dict, tim)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

tim = tims[-1]
cols_hist_origin(ax, sim, ssp_dict, tim)

In [ ]:
run_gif = True

sim = "m12m"
if run_gif:
    gif_path = "../data/results/" + sim + "_tcolor_situ.gif"
    gif_wrapping(
        plot_func=cols_hist_origin,
        param_name="tim",
        param_values=tims,
        gif_path=gif_path,
        fixed_kwargs=dict(sim=sim, ssp_dict=ssp_dict),
        interval=250,
    )

# BEST FITS?

In [ ]:
# # ============================================================
# # GC COLOUR POPULATION STRUCTURE AT A GIVEN SNAPSHOT
# # ============================================================

# # -------------------------------
# # USER INPUTS
# # -------------------------------

# sim = "m12f"
# tim_index = -1  # snapshot index
# bins = np.arange(-2.0, 2.1, 0.1)
# K_test = [1, 2, 3, 4]
# BIC_THRESHOLD = 6.0

# # -------------------------------
# # 1. EXTRACT GC COLOURS
# # -------------------------------

# tidx = tim_index
# all_cols = []

# for it_id, it_data in ssp_dict[sim].items():
#     if it_id == "times":
#         continue

#     mags = np.asarray(it_data["mags"])  # (N_gc, N_time, N_band)

#     # Sanity check once
#     if mags.ndim != 3:
#         raise ValueError(f"Unexpected mags shape for {it_id}: {mags.shape}")

#     # Extract g and z at snapshot tidx
#     g = mags[:, tidx, 0]
#     z = mags[:, tidx, 1]

#     mask = np.isfinite(g) & np.isfinite(z)
#     cols = (g - z)[mask]

#     if cols.size >= 2:
#         all_cols.append(cols)

# R = len(all_cols)
# if R < 5:
#     raise RuntimeError("Not enough realisations for robust statistics")

# print(f"Loaded {R} realisations")

# # -------------------------------
# # 2. AVERAGED HISTOGRAM
# # -------------------------------

# hist_sum = np.zeros(len(bins) - 1)

# for cols in all_cols:
#     h, _ = np.histogram(cols, bins=bins)
#     hist_sum += h

# binsum = hist_sum / R
# bin_centers = 0.5 * (bins[:-1] + bins[1:])
# mean_ngc = np.mean([len(c) for c in all_cols])
# bin_width = bins[1] - bins[0]

# # -------------------------------
# # 3. BIC-BASED STRUCTURE TEST
# # -------------------------------

# x_pool = np.concatenate(all_cols).reshape(-1, 1)

# bics = {}
# gmms = {}

# for K in K_test:
#     gmm = GaussianMixture(
#         n_components=K,
#         covariance_type="full",
#         n_init=20,
#         random_state=42,
#     )
#     gmm.fit(x_pool)
#     bics[K] = gmm.bic(x_pool)
#     gmms[K] = gmm
#     print(f"K={K}, BIC={bics[K]:.2f}")

# # -------------------------------
# # 4. CLASSIFY STRUCTURE
# # -------------------------------

# structure = "unimodal"
# Kbest = 1

# if (bics[1] - bics[2]) >= BIC_THRESHOLD:
#     structure = "bimodal"
#     Kbest = 2

# if 2 in bics and 3 in bics and (bics[2] - bics[3]) >= BIC_THRESHOLD:
#     structure = "trimodal"
#     Kbest = 3

# print(f"\nGC colour population is classified as: {structure.upper()}")

# # -------------------------------
# # 5. BUILD BEST-FIT MODEL CURVE
# # -------------------------------

# gmm = gmms[Kbest]
# weights = gmm.weights_
# means = gmm.means_.flatten()
# sigmas = np.sqrt(gmm.covariances_.flatten())

# x_plot = np.linspace(bins.min(), bins.max(), 1000)

# pdf_total = np.zeros_like(x_plot)
# pdf_components = []

# for k in range(Kbest):
#     pdf_k = weights[k] * norm.pdf(x_plot, means[k], sigmas[k])
#     pdf_components.append(pdf_k)
#     pdf_total += pdf_k

# model_counts = pdf_total * bin_width * mean_ngc

# # -------------------------------
# # 6. FINAL PLOT
# # -------------------------------

# plt.figure(figsize=(7, 5))

# plt.step(
#     bins[:-1],
#     binsum,
#     where="post",
#     color="k",
#     linewidth=2,
#     label="Averaged GC population",
# )

# plt.plot(
#     x_plot,
#     model_counts,
#     color="crimson",
#     linewidth=2.5,
#     label=f"Best-fit GMM ({structure})",
# )

# for k, pdf_k in enumerate(pdf_components):
#     plt.plot(
#         x_plot,
#         pdf_k * bin_width * mean_ngc,
#         linestyle="--",
#         linewidth=1.5,
#         label=f"Component {k + 1}",
#     )

# plt.xlabel(r"$(g - z)_0$")
# plt.ylabel("Mean GC count per bin")
# plt.legend(frameon=False)
# plt.tight_layout()
# plt.show()

# BEST FITS NEW

In [ ]:
def cols_hist_fits(ax, sim, ssp_dict, tim, K_test=[1, 2, 3, 4], BIC_THRESHOLD=6.0, cmap="rainbow"):
    ax.clear()

    bins = np.arange(-2.0, 2.1, 0.1)

    tims = ssp_dict[sim]["times"]
    tidx = np.where(tims == tim)[0][0]

    all_cols = []
    for it_id, it_data in ssp_dict[sim].items():
        if it_id == "times":
            continue

        mags = np.asarray(it_data["mags"])  # (N_gc, N_time, N_band)

        if mags.ndim != 3:
            raise ValueError(f"Unexpected mags shape for {it_id}: {mags.shape}")

        g = mags[:, tidx, 0]
        z = mags[:, tidx, 1]

        mask = np.isfinite(g) & np.isfinite(z)
        cols = (g - z)[mask]

        if cols.size >= 2:
            all_cols.append(cols)

    R = len(all_cols)
    if R < 5:
        raise RuntimeError("Not enough realisations for robust statistics")

    hist_sum = np.zeros(len(bins) - 1)
    for cols in all_cols:
        h, _ = np.histogram(cols, bins=bins)
        hist_sum += h

    binsum = hist_sum / R
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    mean_ngc = np.mean([len(c) for c in all_cols])
    bin_width = bins[1] - bins[0]

    # -------------------------------
    # 2. AVERAGED HISTOGRAM
    # -------------------------------

    hist_sum = np.zeros(len(bins) - 1)

    for cols in all_cols:
        h, _ = np.histogram(cols, bins=bins)
        hist_sum += h

    binsum = hist_sum / R
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    mean_ngc = np.mean([len(c) for c in all_cols])
    bin_width = bins[1] - bins[0]

    # -------------------------------
    # 3. BIC-BASED STRUCTURE TEST
    # -------------------------------

    x_pool = np.concatenate(all_cols).reshape(-1, 1)

    bics = {}
    gmms = {}

    for K in K_test:
        gmm = GaussianMixture(
            n_components=K,
            covariance_type="full",
            n_init=20,
            random_state=42,
        )
        gmm.fit(x_pool)
        bics[K] = gmm.bic(x_pool)
        gmms[K] = gmm

    # -------------------------------
    # 4. CLASSIFY STRUCTURE (GENERAL)
    # -------------------------------

    K_sorted = sorted(bics.keys())
    Kbest = K_sorted[0]

    for K_prev, K_next in zip(K_sorted[:-1], K_sorted[1:]):
        delta_bic = bics[K_prev] - bics[K_next]
        if delta_bic >= BIC_THRESHOLD:
            Kbest = K_next
        else:
            break

    # Human-readable label
    if Kbest == 1:
        structure = "unimodal"
    elif Kbest == 2:
        structure = "bimodal"
    elif Kbest == 3:
        structure = "trimodal"
    else:
        structure = f"{Kbest}-modal"

    print(f"\nGC colour population is classified as: {structure.upper()}")

    # -------------------------------
    # 5. BUILD BEST-FIT MODEL CURVE
    # -------------------------------

    gmm = gmms[Kbest]
    weights = gmm.weights_
    means = gmm.means_.flatten()
    sigmas = np.sqrt(gmm.covariances_.flatten())

    x_plot = np.linspace(bins.min(), bins.max(), 1000)

    pdf_total = np.zeros_like(x_plot)
    pdf_components = []

    for k in range(Kbest):
        pdf_k = weights[k] * norm.pdf(x_plot, means[k], sigmas[k])
        pdf_components.append(pdf_k)
        pdf_total += pdf_k

    model_counts = pdf_total * bin_width * mean_ngc

    # -------------------------------
    # 6. PLOTTING
    # -------------------------------

    cmap = cm.get_cmap(cmap, Kbest)  # or any other colormap

    ax.step(
        bins[:-1],
        binsum,
        where="post",
        color="k",
        linewidth=2,
    )

    ax.plot(
        x_plot,
        model_counts,
        color="k",
        linewidth=2.5,
    )

    for k, pdf_k in enumerate(pdf_components):
        ax.plot(x_plot, pdf_k * bin_width * mean_ngc, linestyle="--", linewidth=1.5, color=cmap(k))

    ax.set_xlabel(r"(g - z)$_{0}$")
    ax.set_ylabel("Number")

    ax.set_xlim(-2, 2)
    ax.set_ylim(0)

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(24, 6))

# match scaling between

tim = tims[-1]
cols_hist(axs[0], sim, ssp_dict, tim)
cols_hist_origin(axs[1], sim, ssp_dict, tim)
cols_hist_fits(axs[1], sim, ssp_dict, tim)

In [ ]:
# # -------------------------------
# # USER INPUTS
# # -------------------------------

# sim = "m12i"
# tim_index = -1  # snapshot index
# bins = np.arange(-2.0, 2.1, 0.1)
# K_test = [1, 2, 3, 4]  # can be any length
# BIC_THRESHOLD = 6.0

# # -------------------------------
# # 1. EXTRACT GC COLOURS
# # -------------------------------

# tidx = tim_index
# all_cols = []

# for it_id, it_data in ssp_dict[sim].items():
#     if it_id == "times":
#         continue

#     mags = np.asarray(it_data["mags"])  # (N_gc, N_time, N_band)

#     if mags.ndim != 3:
#         raise ValueError(f"Unexpected mags shape for {it_id}: {mags.shape}")

#     g = mags[:, tidx, 0]
#     z = mags[:, tidx, 1]

#     mask = np.isfinite(g) & np.isfinite(z)
#     cols = (g - z)[mask]

#     if cols.size >= 2:
#         all_cols.append(cols)

# R = len(all_cols)
# if R < 5:
#     raise RuntimeError("Not enough realisations for robust statistics")

# print(f"Loaded {R} realisations")

# # -------------------------------
# # 2. AVERAGED HISTOGRAM
# # -------------------------------

# hist_sum = np.zeros(len(bins) - 1)

# for cols in all_cols:
#     h, _ = np.histogram(cols, bins=bins)
#     hist_sum += h

# binsum = hist_sum / R
# bin_centers = 0.5 * (bins[:-1] + bins[1:])
# mean_ngc = np.mean([len(c) for c in all_cols])
# bin_width = bins[1] - bins[0]

# # -------------------------------
# # 3. BIC-BASED STRUCTURE TEST
# # -------------------------------

# x_pool = np.concatenate(all_cols).reshape(-1, 1)

# bics = {}
# gmms = {}

# for K in K_test:
#     gmm = GaussianMixture(
#         n_components=K,
#         covariance_type="full",
#         n_init=20,
#         random_state=42,
#     )
#     gmm.fit(x_pool)
#     bics[K] = gmm.bic(x_pool)
#     gmms[K] = gmm
#     print(f"K={K}, BIC={bics[K]:.2f}")

# # -------------------------------
# # 4. CLASSIFY STRUCTURE (GENERAL)
# # -------------------------------

# K_sorted = sorted(bics.keys())
# Kbest = K_sorted[0]

# for K_prev, K_next in zip(K_sorted[:-1], K_sorted[1:]):
#     delta_bic = bics[K_prev] - bics[K_next]
#     if delta_bic >= BIC_THRESHOLD:
#         Kbest = K_next
#     else:
#         break

# # Human-readable label
# if Kbest == 1:
#     structure = "unimodal"
# elif Kbest == 2:
#     structure = "bimodal"
# elif Kbest == 3:
#     structure = "trimodal"
# else:
#     structure = f"{Kbest}-modal"

# print(f"\nGC colour population is classified as: {structure.upper()}")

# # -------------------------------
# # 5. BUILD BEST-FIT MODEL CURVE
# # -------------------------------

# gmm = gmms[Kbest]
# weights = gmm.weights_
# means = gmm.means_.flatten()
# sigmas = np.sqrt(gmm.covariances_.flatten())

# x_plot = np.linspace(bins.min(), bins.max(), 1000)

# pdf_total = np.zeros_like(x_plot)
# pdf_components = []

# for k in range(Kbest):
#     pdf_k = weights[k] * norm.pdf(x_plot, means[k], sigmas[k])
#     pdf_components.append(pdf_k)
#     pdf_total += pdf_k

# model_counts = pdf_total * bin_width * mean_ngc

# # -------------------------------
# # 6. FINAL PLOT
# # -------------------------------

# plt.figure(figsize=(7, 5))

# plt.step(
#     bins[:-1],
#     binsum,
#     where="post",
#     color="k",
#     linewidth=2,
#     label="Averaged GC population",
# )

# plt.plot(
#     x_plot,
#     model_counts,
#     color="crimson",
#     linewidth=2.5,
#     label=f"Best-fit GMM ({structure})",
# )


# cmap = cm.get_cmap("tab10", Kbest)  # or any other colormap

# for k, pdf_k in enumerate(pdf_components):
#     plt.plot(
#         x_plot,
#         pdf_k * bin_width * mean_ngc,
#         linestyle="--",
#         linewidth=1.5,
#         color=cmap(k),
#         label=f"Component {k + 1}",
#     )


# plt.xlabel(r"$(g - z)_0$")
# plt.ylabel("Mean GC count per bin")
# plt.legend(frameon=False)
# plt.tight_layout()
# plt.show()